# Neuro-Symbolic FOL Translation Evaluation Datasets

## Overview

This notebook demonstrates the creation of standardized datasets for evaluating neuro-symbolic First-Order Logic (FOL) translation pipelines.

### Datasets

1. **RuleTaker** - Near-propositional fragment with rule-based reasoning and entailment/not-entailment labels
2. **FOLIO** - Natural language premises with gold-standard FOL annotations for evaluating translation quality
3. **CLUTRR** - Multi-hop kinship reasoning stories with relation queries

### Output Schema

All datasets are standardized to `exp_sel_data_out.json` schema with fields:
- `input` (string): The input text (context + question)
- `output` (string): The expected output/answer
- `metadata_*` fields: Dataset-specific metadata (config, fol_gold, split, example_id)

### Demo Scope

This demo uses a curated subset (`mini_demo_data.json`) with 3 examples from RuleTaker to demonstrate the data processing pipeline. The full dataset contains 56,001 examples across all three datasets.

## Install Dependencies

Following the aii-colab pattern for Colab compatibility. The `loguru` package is not pre-installed on Colab, so it's installed unconditionally. Core scientific packages (numpy, pandas, etc.) would be behind a `google.colab` guard if needed.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru is NOT pre-installed on Colab, always install
_pip('loguru')

# pandas and matplotlib ARE pre-installed on Colab, install locally only
if 'google.colab' not in sys.modules:
    _pip('pandas==2.2.2', 'matplotlib==3.10.0')

## Imports

Standard library and required packages for data loading, processing, and logging.

In [ ]:
import json
from pathlib import Path
from loguru import logger

# Setup logging - log to file and console
logger.remove()
logger.add("logs/data.log", rotation="30 MB", level="DEBUG")
logger.add(lambda msg: print(msg, end=""), level="INFO")

## Data Loading

Load the mini demo data from GitHub (with local fallback). This pattern ensures compatibility with both Colab and local execution.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-27ab00-satisfiability-guided-top-k-ensemble-for/main/round-1/dataset-1/demo/mini_demo_data.json"
import json, os

def load_data():
    """Load data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub load failed: {e}")
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

# Create logs directory if needed
Path("logs").mkdir(exist_ok=True)

In [ ]:
# Load the demo data
data = load_data()
logger.info(f"Loaded data with {len(data['datasets'])} datasets")
for dataset in data['datasets']:
    print(f"Dataset: {dataset['dataset']}, Examples: {len(dataset['examples'])}")

## Configuration

Define tunable parameters for the data processing pipeline.

- `MAX_EXAMPLES_PER_DATASET`: Limit examples per dataset (set to `None` for all)
- `DATASETS_TO_PROCESS`: List of dataset names to process (empty = all)
- `OUTPUT_FILE`: Output filename for standardized data

In [ ]:
# Configuration - MINIMAL values for demo
MAX_EXAMPLES_PER_DATASET = 3  # Process only 3 examples per dataset (minimum to show output)
DATASETS_TO_PROCESS = []  # Empty = process all datasets (or specify: ['ruletaker', 'folio', 'clutrr'])
OUTPUT_FILE = Path("demo_output.json")

print(f"Config: MAX_EXAMPLES_PER_DATASET={MAX_EXAMPLES_PER_DATASET}")
print(f"Config: DATASETS_TO_PROCESS={DATASETS_TO_PROCESS if DATASETS_TO_PROCESS else 'all'}")
print(f"Config: OUTPUT_FILE={OUTPUT_FILE}")

## Process RuleTaker Dataset

RuleTaker contains near-propositional reasoning examples with rule-based entailment labels.

**Original data location**: `temp/datasets/full_ruletaker.json`

**Processing**:
1. Load raw JSON data
2. Extract context and query into `input` field
3. Map `label`/`answer` to `output` field
4. Add metadata (config, example_id)

For this demo, we use the pre-loaded `data` variable instead of reading from file.

In [ ]:
@logger.catch(reraise=True)
def process_ruletaker(data, max_examples=None):
    """Process RuleTaker dataset and standardize to output schema."""
    examples = []
    
    # In demo mode, use pre-loaded data
    # Original code: with open("temp/datasets/full_ruletaker.json") as f:
    #     raw_data = json.load(f)
    
    # Find ruletaker in loaded data
    raw_data = None
    for dataset in data['datasets']:
        if dataset['dataset'] == 'ruletaker':
            raw_data = dataset['examples']
            break
    
    if raw_data is None:
        logger.warning("RuleTaker dataset not found in loaded data")
        return []
    
    for i, ex in enumerate(raw_data):
        if max_examples and i >= max_examples:
            break
        answer = str(ex.get('output', ''))  # In our format, output is directly available
        examples.append({
            "input": ex.get('input', ''),
            "output": answer,
            "metadata_config": ex.get('metadata_config', 'train'),
            "metadata_example_id": ex.get('metadata_example_id', i)
        })
    
    logger.info(f"RuleTaker: {len(examples)} examples processed")
    return examples

# Process RuleTaker
if not DATASETS_TO_PROCESS or 'ruletaker' in DATASETS_TO_PROCESS:
    ruletaker_examples = process_ruletaker(data, max_examples=MAX_EXAMPLES_PER_DATASET)
    print(f"Processed {len(ruletaker_examples)} RuleTaker examples")
    if ruletaker_examples:
        print("\nFirst example:")
        print(f"  Input: {ruletaker_examples[0]['input'][:100]}...")
        print(f"  Output: {ruletaker_examples[0]['output']}")
        print(f"  Config: {ruletaker_examples[0]['metadata_config']}")
else:
    ruletaker_examples = []
    print("Skipping RuleTaker (not in DATASETS_TO_PROCESS)")

## Process FOLIO Dataset

FOLIO contains natural language premises with gold-standard FOL annotations.

**Original data location**: `temp/datasets/full_folio.json`

**Processing**:
1. Load raw JSON data
2. Combine premises into `input` field with "Premises:" prefix
3. Map `answer` to `output` field
4. Add metadata (fol_gold, split, example_id)

In [ ]:
@logger.catch(reraise=True)
def process_folio(data, max_examples=None):
    """Process FOLIO dataset and standardize to output schema."""
    examples = []
    
    # Find folio in loaded data
    raw_data = None
    for dataset in data['datasets']:
        if dataset['dataset'] == 'folio':
            raw_data = dataset['examples']
            break
    
    if raw_data is None:
        logger.warning("FOLIO dataset not found in loaded data")
        return []
    
    for i, ex in enumerate(raw_data):
        if max_examples and i >= max_examples:
            break
        answer = str(ex.get('output', ''))
        examples.append({
            "input": ex.get('input', ''),
            "output": answer,
            "metadata_fol_gold": ex.get('metadata_fol_gold', ''),
            "metadata_split": ex.get('metadata_split', 'train'),
            "metadata_example_id": ex.get('metadata_example_id', f"folio_{i}")
        })
    
    logger.info(f"FOLIO: {len(examples)} examples processed")
    return examples

# Process FOLIO
if not DATASETS_TO_PROCESS or 'folio' in DATASETS_TO_PROCESS:
    folio_examples = process_folio(data, max_examples=MAX_EXAMPLES_PER_DATASET)
    print(f"Processed {len(folio_examples)} FOLIO examples")
    if folio_examples:
        print("\nFirst example:")
        print(f"  Input: {folio_examples[0]['input'][:100]}...")
        print(f"  Output: {folio_examples[0]['output']}")
        print(f"  Split: {folio_examples[0]['metadata_split']}")
else:
    folio_examples = []
    print("Skipping FOLIO (not in DATASETS_TO_PROCESS)")

## Process CLUTRR Dataset

CLUTRR contains multi-hop kinship reasoning stories with relation queries.

**Original data location**: `temp/datasets/full_clutrr.json`

**Processing**:
1. Load raw JSON data
2. Combine story and query into `input` field
3. Map `target_text`/`answer` to `output` field
4. Add metadata (example_id)

In [ ]:
@logger.catch(reraise=True)
def process_clutrr(data, max_examples=None):
    """Process CLUTRR dataset and standardize to output schema."""
    examples = []
    
    # Find clutrr in loaded data
    raw_data = None
    for dataset in data['datasets']:
        if dataset['dataset'] == 'clutrr':
            raw_data = dataset['examples']
            break
    
    if raw_data is None:
        logger.warning("CLUTRR dataset not found in loaded data")
        return []
    
    for i, ex in enumerate(raw_data):
        if max_examples and i >= max_examples:
            break
        examples.append({
            "input": ex.get('input', ''),
            "output": str(ex.get('output', '')),
            "metadata_example_id": ex.get('metadata_example_id', f"clutrr_{i}")
        })
    
    logger.info(f"CLUTRR: {len(examples)} examples processed")
    return examples

# Process CLUTRR
if not DATASETS_TO_PROCESS or 'clutrr' in DATASETS_TO_PROCESS:
    clutrr_examples = process_clutrr(data, max_examples=MAX_EXAMPLES_PER_DATASET)
    print(f"Processed {len(clutrr_examples)} CLUTRR examples")
    if clutrr_examples:
        print("\nFirst example:")
        print(f"  Input: {clutrr_examples[0]['input'][:100]}...")
        print(f"  Output: {clutrr_examples[0]['output']}")
else:
    clutrr_examples = []
    print("Skipping CLUTRR (not in DATASETS_TO_PROCESS)")

## Save Standardized Output

Combine all processed datasets and save to JSON file in the standardized schema format.

In [ ]:
# Combine all datasets
datasets = []

if ruletaker_examples:
    datasets.append({"dataset": "ruletaker", "examples": ruletaker_examples})
if folio_examples:
    datasets.append({"dataset": "folio", "examples": folio_examples})
if clutrr_examples:
    datasets.append({"dataset": "clutrr", "examples": clutrr_examples})

# Save output
output = {"datasets": datasets}
with open(OUTPUT_FILE, 'w') as f:
    json.dump(output, f, indent=2)

total_examples = sum(len(d['examples']) for d in datasets)
logger.info(f"Saved to {OUTPUT_FILE}")
logger.info(f"Total datasets: {len(datasets)}")
logger.info(f"Total examples: {total_examples}")

## Results Visualization

Display summary statistics and sample data from the processed datasets.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the saved output to verify
with open(OUTPUT_FILE) as f:
    result = json.load(f)

# Create summary table
summary_data = []
for dataset in result['datasets']:
    examples = dataset['examples']
    summary_data.append({
        'Dataset': dataset['dataset'],
        'Num Examples': len(examples),
        'Avg Input Length': sum(len(e['input']) for e in examples) / len(examples) if examples else 0,
        'Output Types': len(set(e['output'] for e in examples))
    })

df_summary = pd.DataFrame(summary_data)
print("=== Dataset Summary ===")
print(df_summary.to_string(index=False))
print()

# Visualize dataset sizes
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart of example counts
ax1 = axes[0]
datasets_names = [d['dataset'] for d in result['datasets']]
example_counts = [len(d['examples']) for d in result['datasets']]
ax1.bar(datasets_names, example_counts, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
ax1.set_ylabel('Number of Examples')
ax1.set_title('Dataset Sizes (Demo)')
ax1.set_ylim(0, max(example_counts) * 1.2)
for i, v in enumerate(example_counts):
    ax1.text(i, v + 0.1, str(v), ha='center')

# Show sample outputs
ax2 = axes[1]
ax2.axis('off')
sample_text = "Sample Outputs:\n\n"
for dataset in result['datasets'][:1]:  # Show first dataset samples
    sample_text += f"{dataset['dataset'].upper()}:\n"
    for i, ex in enumerate(dataset['examples'][:2]):
        sample_text += f"  Ex {i+1}: {ex['output']}\n"
ax2.text(0.1, 0.5, sample_text, fontsize=10, family='monospace',
         transform=ax2.transAxes, verticalalignment='center')

plt.tight_layout()
plt.show()

# Display full sample from each dataset
print("\n=== Full Sample from Each Dataset ===")
for dataset in result['datasets']:
    print(f"\n--- {dataset['dataset'].upper()} ---")
    if dataset['examples']:
        ex = dataset['examples'][0]
        print(f"Input: {ex['input'][:150]}...")
        print(f"Output: {ex['output']}")
        # Show available metadata fields
        metadata_fields = {k: v for k, v in ex.items() if k.startswith('metadata_')}
        print(f"Metadata: {metadata_fields}")